# Lab 3 — A tiny transformer with causal and masked objectives

COMP-672 · John Glossner · 2 points

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glossner/COMP-672-Labs/blob/main/labs/week03_starter.ipynb)

**First: File → Save a copy in Drive.** Sign in to Google to run and save. Rename your copy, for example `COMP672_Lab03_YourName.ipynb`, and work in that copy. No GitHub account is needed for this path.

Opening this notebook does not execute code, save your answers to GitHub, or share your work. Use the cell play buttons yourself. The functions marked `NotImplementedError` are exercises for you to complete. The supplied checks and examples are scaffolding, not an exhaustive grader.

Use a **CPU** runtime. All data are original synthetic instructional examples generated by the code below; no datasets, models, credentials, Drive mount, or repository clone are required.

[Course lab index and submission instructions](https://github.com/glossner/COMP-672-Labs/blob/main/README.md) · [Handout](https://github.com/glossner/COMP-672-Labs/blob/main/labs/LAB03.md)


## Install the lab environment

Use Colab’s **default Python CPU runtime**; you do not need to select an older Python runtime. This revision supports Python **3.11–3.14**, including Python 3.13 in Google’s documented Colab backend snapshot. Python 3.14 support is also tested for local/Jupyter environments; this does not mean Colab offers a Python 3.14 runtime. Run this cell before the lab imports. It prints your actual Python version and installs only missing or mismatched pinned packages from PyPI (and the official CPU PyTorch index for Labs 3 and 5). Python 3.14 selects newer NumPy, scikit-learn, and Matplotlib pins; Python 3.11–3.13 keeps the established pins. Binary wheels are required so setup will not try to compile scientific packages.

If Colab prompts for a restart after installation, select **Runtime → Restart session**, then rerun from this cell. Do not change the runtime’s system Python with `apt`, `conda`, or `update-alternatives`. A fresh runtime is preferable if you already installed unrelated packages.

Nothing is installed or run until you execute a cell. Files in the runtime are temporary; save your notebook and download your results before disconnecting.

**Updating an existing Drive copy:** GitHub changes do not update your copy automatically. Keep your current copy as a backup. Reopen this lab from the course README and save a new copy in Drive, then transfer your work; or replace only the two setup/environment code cells and these setup instructions in your existing copy with the matching lab’s latest cells. Restart the session and rerun setup. Do not overwrite your completed exercise or analysis cells.


In [ ]:
import sys
import subprocess
import importlib.metadata as metadata

if not (3, 11) <= sys.version_info[:2] <= (3, 14):
    raise RuntimeError("This lab supports Python 3.11–3.14. Reopen the latest notebook from the course README if your runtime is newer.")

print("Runtime Python:", sys.version.split()[0])

LAB_PACKAGES = {'numpy': '2.1.3', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'matplotlib': '3.10.0'}
# Python 3.14 needs newer binary wheels; retain the established pins on 3.11–3.13.
if sys.version_info[:2] >= (3, 14):
    LAB_PACKAGES.update({'numpy': '2.3.5', 'scikit-learn': '1.7.2', 'matplotlib': '3.10.8'})
LAB_PACKAGES["torch"] = "2.11.0"
def installed_version(name):
    try:
        return metadata.version(name).split("+")[0]
    except metadata.PackageNotFoundError:
        return None

needed = [f"{name}=={version}" for name, version in LAB_PACKAGES.items()
          if name != "torch" and installed_version(name) != version]
if needed:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--only-binary=:all:", *needed])
if installed_version("torch") != LAB_PACKAGES["torch"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--only-binary=:all:",
                           "torch==2.11.0", "--index-url",
                           "https://download.pytorch.org/whl/cpu"])
print("Package setup complete. Restart the session if Colab requests it.")


In [ ]:
import platform
import importlib

LAB_VERSIONS = {"python": platform.python_version()}
for distribution, module_name in [("numpy", "numpy"), ("scipy", "scipy"),
                                  ("scikit-learn", "sklearn"), ("matplotlib", "matplotlib")]:
    module = importlib.import_module(module_name)
    LAB_VERSIONS[distribution] = module.__version__
import torch
torch.set_num_threads(1)
LAB_VERSIONS["torch"] = torch.__version__
for name, expected in LAB_PACKAGES.items():
    if LAB_VERSIONS[name].split("+")[0] != expected:
        raise RuntimeError(f"Restart the session: {name} is already imported at a different version.")
print(LAB_VERSIONS)
print("CPU environment ready. Synthetic data will be generated by the lab functions.")


# Lab 3 — A tiny transformer with causal and masked objectives

**2 points.** Chapters 7, 8, 9; outcomes LO3, LO8. Allow 4–5 hours including the 60-minute lab meeting. Complete the Python starter or the Jupyter starter, not both. The scaffolding contains intentional `NotImplementedError` gaps.

## Implementation tasks

1. Implement scaled attention and a causal mask; verify that changing a future value leaves earlier causal outputs unchanged.

2. Implement shifted next-token cross-entropy and explain why the last position is excluded.

3. Implement a selected-position masked loss and contrast it with the causal objective; ensure unselected positions do not contribute.

4. Implement the rank-2 adapter forward pass with a frozen base head and scale alpha/r. Count trainable parameters and check the zero-update initialization.


Functions to complete: `attention`, `causal_loss`, `masked_loss`, `LoRALinear.forward`.

## Required experiments

- Run the tiny causal pretraining and head-only LoRA adaptation experiments. Report initial loss, fixed-training-endpoint test loss, adapter parameter count, and adaptation loss before and after. Use the same adapted-distribution test set for the before/after comparison.

- Run the bidirectional masked model. Perturb right context to demonstrate visibility and explain why its loss is not directly ranked against causal perplexity.

## Brief analysis in your notebook or code comments

1. Which causal-mask bug could create deceptively low training loss?

2. What can adaptation of only an output head change, and what can it not directly change in the frozen backbone?

3. The finite grammar repeats possible sequences across independent draws. Explain why these results measure a small distribution-fitting exercise rather than held-out linguistic compositionality.


Answer each question in approximately 2–4 sentences beside the results. This is part of the lab, with no separate report.

## Submission and rubric

Submit the link to your completed, saved Colab copy (or edited notebook/source if working locally), a results JSON file, and an execution command. Follow the save/share steps in the repository README. For Colab, state “Run cells from top to bottom in a fresh CPU runtime.” Include versions, seed, split policy, settings, runtime, and clear labels. No points depend on a positive gain.

| Criterion | Points | Evidence |
| --- | --- | --- |
| Implementation | 0.8 | Required functions implement the intended mathematics and boundary behavior |
| Experiments | 0.6 | Both required comparisons are run and outputs are recorded |
| Comprehension | 0.4 | Brief answers accurately connect results to the three analysis questions |
| Reproducibility | 0.2 | Runnable submission with configuration, seed, and data scope |
| Total | 2.0 | Partial credit is proportional to the evidence supplied |

Split implementation credit equally among required functions; split experiment credit equally between the two comparisons. Assess the three analysis answers together on a 0–0.4 scale; award 0.4 for all three accurate and grounded, 0.2 for partial understanding, 0 for absent or fundamentally incorrect explanations. Intermediate credit is allowed.


## Starter code

Complete the exercise functions below. Keep the supplied checks and add the cases required by the handout.

In [ ]:
"""Week 3 student starter: tiny transformer; no downloaded weights or corpora."""


In [ ]:
import math


In [ ]:
import torch


In [ ]:
from torch import nn


In [ ]:
from torch.nn import functional as F


In [ ]:
torch.set_num_threads(1)


In [ ]:
def attention(q,k,v,causal=True):
    """Inputs B,H,T,d. Return attended values and row-normalized weights."""
    raise NotImplementedError("Complete this laboratory task.")


In [ ]:
def causal_loss(logits,tokens):
    """logits at t predict token t+1; all sequences are equal length."""
    raise NotImplementedError("Complete this laboratory task.")


In [ ]:
def masked_loss(logits,targets,selected):
    """selected is a Boolean B,T mask. No loss on unselected positions."""
    raise NotImplementedError("Complete this laboratory task.")


In [ ]:
class TinyTransformer(nn.Module):
    def __init__(self,vocab=12,width=24,heads=3):
        super().__init__(); self.heads=heads; self.width=width
        self.embed=nn.Embedding(vocab,width); self.pos=nn.Embedding(12,width)
        self.qkv=nn.Linear(width,3*width); self.proj=nn.Linear(width,width)
        self.norm1=nn.LayerNorm(width); self.norm2=nn.LayerNorm(width)
        self.ff=nn.Sequential(nn.Linear(width,48),nn.ReLU(),nn.Linear(48,width))
        self.head=nn.Linear(width,vocab)
    def forward(self,ids,causal=True):
        b,t=ids.shape; d=self.width//self.heads
        x=self.embed(ids)+self.pos(torch.arange(t,device=ids.device))
        q,k,v=self.qkv(self.norm1(x)).chunk(3,-1)
        q,k,v=[z.view(b,t,self.heads,d).transpose(1,2) for z in (q,k,v)]
        z,_=attention(q,k,v,causal)
        x=x+self.proj(z.transpose(1,2).contiguous().view(b,t,self.width))
        x=x+self.ff(self.norm2(x)); return self.head(x)


In [ ]:
class LoRALinear(nn.Module):
    def __init__(self,base,rank=2,alpha=2):
        super().__init__(); self.base=base; self.scale=alpha/rank
        for p in base.parameters(): p.requires_grad=False
        self.A=nn.Parameter(torch.randn(rank,base.in_features)*.02)
        self.B=nn.Parameter(torch.zeros(base.out_features,rank))
    def forward(self,x):
        raise NotImplementedError("Complete this laboratory task.")


In [ ]:
def data(n,seed,shift=0):
    g=torch.Generator().manual_seed(seed)
    a=torch.randint(1,5,(n,),generator=g); b=torch.randint(5,9,(n,),generator=g)
    # 0 BOS, 9 separator, 10 EOS, 11 MASK; deterministic repeated structure.
    return torch.stack([a*0,a,b,a*0+9,((a-1+shift)%4)+1,b,a*0+10],1)


In [ ]:
def checks():
    torch.manual_seed(17)
    q=torch.randn(2,3,4,8); k=torch.randn_like(q); v=torch.randn_like(q)
    z,w=attention(q,k,v,True)
    assert torch.allclose(w.sum(-1),torch.ones_like(w.sum(-1)))
    assert torch.count_nonzero(w.triu(1))==0
    changed=v.clone(); changed[:,:,3]+=100
    assert torch.allclose(attention(q,k,changed,True)[0][:,:,:3],z[:,:,:3])
    base=nn.Linear(4,3); layer=LoRALinear(base)
    x=torch.randn(2,4); assert torch.allclose(layer(x),base(x))
    logits=torch.zeros(1,3,12); selected=torch.tensor([[False,True,False]])
    assert torch.isclose(masked_loss(logits,torch.tensor([[0,1,2]]),selected),torch.tensor(math.log(12)))


In [ ]:
def experiment():
    torch.manual_seed(17); train=data(128,17); dev=data(64,18); test=data(64,19)
    # Samples are independently generated draws of a finite grammar, not a claim of novel grammar generalization.
    model=TinyTransformer(); opt=torch.optim.Adam(model.parameters(),lr=.01)
    initial=causal_loss(model(dev),dev).item()
    for _ in range(100):
        opt.zero_grad(); loss=causal_loss(model(train),train); loss.backward(); opt.step()
    model.eval(); base_test=causal_loss(model(test),test).item()
    adapted=data(64,20,shift=1)
    for p in model.parameters(): p.requires_grad=False
    model.head=LoRALinear(model.head)
    params=[p for p in model.parameters() if p.requires_grad]
    opt=torch.optim.Adam(params,lr=.03)
    adapted_test=data(64,21,shift=1)
    before=causal_loss(model(adapted_test),adapted_test).item()
    for _ in range(80):
        opt.zero_grad(); loss=causal_loss(model(adapted),adapted); loss.backward(); opt.step()
    after=causal_loss(model(adapted_test),adapted_test).item()
    mlm=TinyTransformer(); opt=torch.optim.Adam(mlm.parameters(),lr=.01)
    selected=torch.zeros_like(train,dtype=torch.bool); selected[:,4]=True
    corrupt=train.clone(); corrupt[selected]=11
    for _ in range(80):
        opt.zero_grad(); loss=masked_loss(mlm(corrupt,False),train,selected); loss.backward(); opt.step()
    ts=torch.zeros_like(test,dtype=torch.bool); ts[:,4]=True; tc=test.clone(); tc[ts]=11
    return {'initial_dev_causal_loss':initial,'base_test_causal_loss':base_test,'adapter_trainable_parameters':sum(p.numel() for p in params),'adapted_distribution_loss_before':before,'adapted_test_loss_after':after,'masked_test_loss':masked_loss(mlm(tc,False),test,ts).item(),'warning':'Causal and masked losses score different events; do not rank their model quality from these numbers.'}


## Run checks and the supplied experiment

Complete the exercise functions first. A `NotImplementedError` here is expected in an untouched starter. The result dictionary covers the supplied baseline; the handout also requires comparisons and analysis that you must add below.


In [ ]:
import time

checks()
_started = time.perf_counter()
results = experiment()
baseline_runtime_seconds = time.perf_counter() - _started
results


## Your required comparisons

Add the additional experiments, checks, plots/tables, measured runtimes, and clearly labeled results required by the handout. Add their JSON-compatible summaries to `additional_results`.


In [ ]:
# TODO: implement and run the remaining required comparisons here.
additional_results = {}


## Brief analysis

1. Which causal-mask bug could create deceptively low training loss?

2. What can adaptation of only an output head change, and what can it not directly change in the frozen backbone?

3. The finite grammar repeats possible sequences across independent draws. Explain why these results measure a small distribution-fitting exercise rather than held-out linguistic compositionality.

Write approximately 2–4 sentences for each question in the next cell, tied to your measured results.

1. _Your response:_

2. _Your response:_

3. _Your response:_


## Record configuration and export results

Complete the configuration fields and include all required comparison results. Run the export cell after your experiments. The JSON will also appear in the saved notebook output so a viewer can read it. Download the generated JSON via Colab’s **Files** sidebar (file menu → Download) and attach it with your notebook link in the course submission channel. The runtime file alone is not saved in Drive or shared with your notebook.


In [ ]:
# TODO: document the settings you actually used.
configuration = {
    "seed": None,
    "split_policy": "",
    "settings_and_changes": "",
    "additional_experiment_runtime_seconds": None,
    "execution": "Run cells from top to bottom in a fresh CPU runtime",
}


In [ ]:
import json
from pathlib import Path

def json_ready(value):
    # Support NumPy scalars/arrays and nested result dictionaries.
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if hasattr(value, "tolist"):
        return value.tolist()
    return value

payload = {
    "lab": 3,
    "versions": LAB_VERSIONS,
    "configuration": configuration,
    "baseline_runtime_seconds": baseline_runtime_seconds,
    "results": results,
    "additional_results": additional_results,
}
results_path = Path("week03_results.json")
results_json = json.dumps(json_ready(payload), indent=2, allow_nan=False)
results_path.write_text(results_json + "\n", encoding="utf-8")
print(results_json)
print("Download", results_path.name, "from the Files sidebar before disconnecting.")


## Save and submit

1. Finish the code, required comparisons, and three analysis responses. Run the completed notebook from top to bottom in a fresh CPU runtime and retain the outputs.
2. **File → Save** your Drive copy and wait for saving to finish. Ensure **Edit → Notebook settings → Omit code cell output when saving this notebook** is turned off. Reopen your copy to verify the code, analysis, and results remain visible.
3. Download `week03_results.json` from the Files sidebar. Runtime files disappear when the runtime is deleted.
4. In **your copy**, select **Share → General access → Anyone with the link → Viewer**, then **Copy link** and **Done**. Anyone holding this link can view the full saved notebook, including code, output, and comments. Keep unrelated private information out.
5. Submit **your saved copy’s link** and results JSON through the submission channel specified by John Glossner. Do not submit this blank GitHub starter’s link. Verify the link opens for a signed-out viewer. If your school account blocks link sharing, ask the instructor for an approved alternative.

These actions require you to run, save, and share deliberately; this notebook never does them automatically. Cloning or opening the repository alone does not save your answers. A GitHub fork is optional, requires a GitHub account, and is public because the source repository is public; use Drive for the default course workflow.
